# Hito 3 - Experimentos (Fases 3-5)
Notebook de ejecución para entrenamiento, evaluación y generación de tablas/figuras de MIMIC-TRIAGE.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

from hito3_experiments import run_experiments

ModuleNotFoundError: No module named 'pandas'

In [ ]:
project_root = Path.cwd().parent  # asumido: notebook dentro de 'Hito 3'
manifest = run_experiments(project_root=project_root, random_state=42, n_iter=30, force_rebuild=False)
print(json.dumps(manifest, indent=2, ensure_ascii=False))

In [ ]:
tables_dir = Path(manifest['tables_dir'])
figures_dir = Path(manifest['figures_dir'])

table_6_1 = pd.read_csv(tables_dir / 'table_6_1_split_distribution.csv')
table_6_2 = pd.read_csv(tables_dir / 'table_6_2_auc_brier.csv')
table_6_3 = pd.read_csv(tables_dir / 'table_6_3_recall_at_k.csv')
table_6_4 = pd.read_csv(tables_dir / 'table_6_4_pct_top_k.csv')
table_6_5 = pd.read_csv(tables_dir / 'table_6_5_summary_gains.csv')

display(table_6_1)
display(table_6_2)
display(table_6_3)
display(table_6_4)
display(table_6_5)

In [ ]:
display(Image(filename=str(figures_dir / 'roc_comparison_models.png')))
display(Image(filename=str(figures_dir / 'calibration_comparison_models.png')))

In [ ]:
for image_path in sorted(figures_dir.glob('scores_distribution_*.png')):
    print(image_path.name)
    display(Image(filename=str(image_path)))

## Verificación de reutilización de Hito 2 y coherencia de datos

En esta sección se deja constancia explícita de:

1. **Dataset y features**: si `processed_features_48h_setA.csv` se reutiliza desde Hito 2 o se reconstruye desde RAW con la misma ingeniería 0-48h.
2. **Baseline Hito 2 replicado**: ejecución de un baseline lineal con el mismo criterio de Hito 2 (subset de features, imputación por mediana, estandarización z-score y score probabilístico).
3. **Coherencia documental**: contraste de tamaño y prevalencia frente a lo documentado en Hito 2 (4000 filas, 338 columnas, prevalencia ~13.85%).

In [ ]:
from hito3_experiments import (
    BASELINE_HITO2_FEATURES,
    BASELINE_HITO2,
    resolve_paths,
    load_or_build_dataset,
)

paths = resolve_paths(project_root)
df_check, source_check = load_or_build_dataset(paths, force_rebuild=False)

expected_rows = 4000
expected_cols = 338
expected_prevalence = 0.1385

actual_rows, actual_cols = df_check.shape
actual_prevalence = float(df_check["In-hospital_death"].mean())

summary_check = pd.DataFrame([
    {
        "criterio": "filas",
        "esperado_hito2": expected_rows,
        "observado": actual_rows,
        "coincide": actual_rows == expected_rows,
    },
    {
        "criterio": "columnas",
        "esperado_hito2": expected_cols,
        "observado": actual_cols,
        "coincide": actual_cols == expected_cols,
    },
    {
        "criterio": "prevalencia",
        "esperado_hito2": expected_prevalence,
        "observado": round(actual_prevalence, 4),
        "coincide": round(actual_prevalence, 4) == round(expected_prevalence, 4),
    },
])

print(f"Origen del dataset usado por Hito 3: {source_check}")
print("Interpretación origen: hito2_csv = reutiliza CSV de Hito 2; rebuilt_from_raw = reconstruido desde set-a/outcomes-a con misma ingeniería.")
display(summary_check)

In [ ]:
feature_cols_hito3_full = [c for c in df_check.columns if c not in ["RecordID", "In-hospital_death"]]

feature_compare = pd.DataFrame({
    "feature_hito2_documentada": BASELINE_HITO2_FEATURES,
    "presente_en_hito3": [f in df_check.columns for f in BASELINE_HITO2_FEATURES],
    "valor_no_nulo_pct": [round(100 * df_check[f].notna().mean(), 2) if f in df_check.columns else None for f in BASELINE_HITO2_FEATURES],
})

print(f"Número de features usadas por modelos Hito 3 (full pipeline): {len(feature_cols_hito3_full)}")
print(f"Número de features baseline Hito 2 disponibles en Hito 3: {feature_compare['presente_en_hito3'].sum()} / {len(BASELINE_HITO2_FEATURES)}")

print("\nFeatures baseline Hito 2 esperadas:")
print(BASELINE_HITO2_FEATURES)

display(feature_compare)

In [ ]:
# Corrida más completa tras ajustes (puedes cambiar n_iter a 50 si quieres más exploración)
manifest_full = run_experiments(project_root=project_root, random_state=42, n_iter=30, force_rebuild=False)
print(json.dumps(manifest_full, indent=2, ensure_ascii=False))

print("\nIncluye baseline replicado:", "baseline_hito2_replicated_60_20_20" in manifest_full["models_ran"])
print("Features baseline usadas:", manifest_full.get("baseline_hito2_features_used", []))

In [ ]:
tables_dir_full = Path(manifest_full['tables_dir'])
figures_dir_full = Path(manifest_full['figures_dir'])

display(pd.read_csv(tables_dir_full / 'table_feature_reuse_hito2_vs_hito3.csv'))
display(pd.read_csv(tables_dir_full / 'table_6_2_auc_brier.csv'))
display(pd.read_csv(tables_dir_full / 'table_6_3_recall_at_k.csv'))
display(pd.read_csv(tables_dir_full / 'table_6_4_pct_top_k.csv'))